# Notebook 04 — Hyperparameter Tuning with Time-Series Cross-Validation

## RustWeatherML 

### Why **not** random k-fold?

Standard k-fold shuffles rows, which for a time series *leaks future
information*. In `RustWeatherML` this would be especially bad because
the lags ($T_{t-1}, T_{t-24}$, ...) already contain the temporal
neighbor — any random split would end up training on observations
*adjacent* to the validation ones.

### Forward-chaining (a.k.a. expanding window)

Successive splits where each fold trains on $[1, t_k]$ and validates on
$(t_k, t_{k+1}]$:

```
fold 1: train [    1 ..  500]   val [ 501 .. 1000]
fold 2: train [    1 .. 1000]   val [1001 .. 1500]
fold 3: train [    1 .. 1500]   val [1501 .. 2000]
...
```

This replicates the production scenario: you always predict the future
from the known past.

### What we learned in Notebook 03

| Model | Test RMSE | Test MCC | Comment from Nb03 |
|---|---|---|---|
| **Ridge alpha=1** | **3.529** | — | best regressor; room in alpha |
| OLS | 3.529 | — | same as Ridge -> alpha=1 still weak |
| Lasso alpha=0.1 | 3.598 | — | automatic sparsity |
| RF regressor | 3.657 | — | tree-based loses to linear (rare!) |
| GB regressor | 3.669 | — | val RMSE plateaus around K~60 |
| **RF classifier** | — | **0.719** | best classifier by a wide margin |
| DT classifier | — | 0.637 | clear overfitting |
| Logistic | — | 0.608 | good recall (0.93) |

### Tuning strategy

**Regression**:
1. Ridge — log-space grid search over $\alpha \in \{10^{-3}, ..., 10^3\}$
2. Lasso — log-space grid search over $\alpha \in \{10^{-4}, ..., 10^1\}$
3. RandomForest — grid over $(n_{trees}, \text{max\_depth})$
4. GradientBoosting — random search over $(K, d, \eta)$ with early stopping

**Classification**:
1. RandomForestClassifier — grid over $(n_{trees}, \text{max\_depth})$
2. LogisticRegression — $\alpha$ (regularization)

**Validation**: we use the last CV fold (longest train) for numerical
stability. Smaller folds were ill-conditioned (80 features with small $n$).
**Selection metric**: RMSE for regression, MCC for classification.

In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde", "approx"] }
:dep smartcore = "0.3"
:dep rand = "0.8"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [2]:
use polars::prelude::*;
use ndarray::{Array1, Array2, Axis};
use std::collections::HashMap;
use std::fs::File;

use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};
use smartcore::linear::lasso::{Lasso, LassoParameters};
use smartcore::linear::logistic_regression::{LogisticRegression, LogisticRegressionParameters};
use smartcore::tree::decision_tree_regressor::{DecisionTreeRegressor, DecisionTreeRegressorParameters};
use smartcore::ensemble::random_forest_regressor::{RandomForestRegressor, RandomForestRegressorParameters};
use smartcore::ensemble::random_forest_classifier::{RandomForestClassifier, RandomForestClassifierParameters};

use rand::{rngs::StdRng, SeedableRng, Rng};

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Load preprocessed data

In [3]:
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let val_df   = LazyFrame::scan_parquet("../data/features/val.parquet",   Default::default())
    .unwrap().collect().unwrap();

println!("Train: {} x {}", train_df.height(), train_df.width());
println!("Val:   {} x {}",   val_df.height(),   val_df.width());

Train: 20160 x 114


Val:   5040 x 114


---
## 2. Reuse feature list from Notebook 03 (post-decorrelation)

We read the canonical list from `models/model_comparison.json`, produced
by Notebook 03. This guarantees that tuning uses exactly the same
features as the leaderboard.

In [4]:
let comparison_str = std::fs::read_to_string("../models/model_comparison.json")
    .expect("Notebook 03 must have been executed first");
let comp: serde_json::Value = serde_json::from_str(&comparison_str).unwrap();
let final_features: Vec<String> = comp["feature_names"].as_array().unwrap()
    .iter().map(|v| v.as_str().unwrap().to_string()).collect();
println!("{} features loaded from model_comparison.json", final_features.len());

80 features loaded from model_comparison.json


In [5]:
fn df_to_array2(df: &DataFrame, cols: &[String]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for c in cols {
        let s = df.column(c.as_str()).unwrap();
        let f = s.cast(&DataType::Float64).unwrap();
        let ca = f.f64().unwrap().to_vec();
        for v in ca { data.push(v.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, name: &str) -> Array1<f64> {
    let v: Vec<f64> = df.column(name).unwrap()
        .cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec().into_iter()
        .map(|x| x.unwrap_or(0.0)).collect();
    Array1::from_vec(v)
}

fn ndarray_to_dense(a: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&a.outer_iter().map(|r| r.to_vec()).collect::<Vec<_>>())
}

fn rmse(yt: &[f64], yp: &[f64]) -> f64 {
    let n = yt.len() as f64;
    let s: f64 = yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).powi(2)).sum();
    (s / n).sqrt()
}

fn mcc(yt: &[u32], yp: &[u32]) -> f64 {
    let mut tp = 0i64; let mut tn = 0i64; let mut fp = 0i64; let mut fnn = 0i64;
    for (t, p) in yt.iter().zip(yp.iter()) {
        match (*t, *p) {
            (1,1) => tp += 1,
            (0,0) => tn += 1,
            (0,1) => fp += 1,
            (1,0) => fnn += 1,
            _ => {}
        }
    }
    let denom = ((tp+fp) as f64 * (tp+fnn) as f64 * (tn+fp) as f64 * (tn+fnn) as f64).sqrt();
    if denom > 0.0 { (tp as f64 * tn as f64 - fp as f64 * fnn as f64) / denom } else { 0.0 }
}

println!("Helpers ready.");

Helpers ready.


In [6]:
let train_clean = train_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
    .and(col("temp_lag24h").is_not_null())
).sort(["timestamp"], Default::default()).collect().unwrap();

let val_clean = val_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
    .and(col("temp_lag24h").is_not_null())
).sort(["timestamp"], Default::default()).collect().unwrap();

println!("Valid train: {}", train_clean.height());
println!("Valid val:   {}", val_clean.height());

let X_train = df_to_array2(&train_clean, &final_features);
let X_val   = df_to_array2(&val_clean,   &final_features);
let y_train_temp = df_to_array1(&train_clean, "temp_next_24h");
let y_val_temp   = df_to_array1(&val_clean,   "temp_next_24h");
let y_train_rain: Vec<u32> = df_to_array1(&train_clean, "will_rain_next_24h").iter().map(|&v| v as u32).collect();
let y_val_rain:   Vec<u32> = df_to_array1(&val_clean,   "will_rain_next_24h").iter().map(|&v| v as u32).collect();

// Standardization -- fit on the full train (CV will use subsets).
fn fit_scaler(x: &Array2<f64>) -> (Vec<f64>, Vec<f64>) {
    let n_feat = x.ncols();
    let mut means = vec![0.0_f64; n_feat];
    let mut stds  = vec![1.0_f64; n_feat];
    for j in 0..n_feat {
        let c = x.column(j);
        let mu = c.mean().unwrap_or(0.0);
        let var: f64 = c.iter().map(|v| (v - mu).powi(2)).sum::<f64>() / (c.len() as f64 - 1.0).max(1.0);
        means[j] = mu;
        stds[j]  = var.sqrt().max(1e-8);
    }
    (means, stds)
}

fn standardize(x: &Array2<f64>, means: &[f64], stds: &[f64]) -> Array2<f64> {
    let mut o = x.clone();
    for j in 0..x.ncols() {
        for i in 0..x.nrows() {
            o[[i, j]] = (x[[i, j]] - means[j]) / stds[j];
        }
    }
    o
}

let (means, stds) = fit_scaler(&X_train);
let X_train_z = standardize(&X_train, &means, &stds);
let X_val_z   = standardize(&X_val,   &means, &stds);

println!("Matrices ready: X_train {:?}, X_val {:?}", X_train.shape(), X_val.shape());

Valid train: 19488


Valid val:   5040


Matrices ready: X_train [19488, 80], X_val [5040, 80]


---
## 3. `TimeSeriesSplit` (forward-chaining)

Manual implementation: given $n$ samples and $k$ folds we produce $k$
splits where fold $i$ trains on $[0, t_i]$ and validates on
$(t_i, t_{i+1}]$, with $t_i = \lfloor i \cdot n/(k+1) \rfloor$.

In [7]:
fn time_series_splits(n: usize, k: usize) -> Vec<(Vec<usize>, Vec<usize>)> {
    let mut out = Vec::with_capacity(k);
    let block = n / (k + 1);
    for i in 1..=k {
        let train_end = block * i;
        let val_end   = (block * (i + 1)).min(n);
        let train_idx: Vec<usize> = (0..train_end).collect();
        let val_idx:   Vec<usize> = (train_end..val_end).collect();
        if !val_idx.is_empty() { out.push((train_idx, val_idx)); }
    }
    out
}

// We use 3 folds (instead of 5) to fit the notebook time budget. The Lasso
// in smartcore has O(n^2 p) complexity and was the bottleneck.
let folds = time_series_splits(X_train.nrows(), 3);
println!("CV folds (forward-chaining, k=3):");
for (i, (tr, va)) in folds.iter().enumerate() {
    println!("  fold {}: train {} samples, val {} samples", i+1, tr.len(), va.len());
}

CV folds (forward-chaining, k=3):


  fold 1: train 4872 samples, val 4872 samples


  fold 2: train 9744 samples, val 4872 samples


  fold 3: train 14616 samples, val 4872 samples


()

In [8]:
fn select_rows(x: &Array2<f64>, idx: &[usize]) -> Array2<f64> {
    let n_cols = x.ncols();
    let mut data = Vec::with_capacity(idx.len() * n_cols);
    for &i in idx {
        for j in 0..n_cols { data.push(x[[i, j]]); }
    }
    Array2::from_shape_vec((idx.len(), n_cols), data).unwrap()
}

fn select_y(y: &Array1<f64>, idx: &[usize]) -> Vec<f64> {
    idx.iter().map(|&i| y[i]).collect()
}

fn select_yu(y: &[u32], idx: &[usize]) -> Vec<u32> {
    idx.iter().map(|&i| y[i]).collect()
}

println!("Subset helpers ready.");

Subset helpers ready.


---
## 4. Grid search — Ridge

NOTE: we use **only the last fold** (the longest train, approx 14k samples)
to tune Ridge. Smaller folds were numerically unstable because $X^TX$
becomes ill-conditioned when $n \sim 2p$. The last fold is the closest
proxy to the production scenario and gives stable estimates.

In [9]:
let alphas = [0.01_f64, 0.1, 1.0, 10.0, 100.0, 1000.0];
let mut ridge_results: Vec<(f64, f64)> = Vec::new();

let last = folds.len() - 1;
let tr_idx_r = folds[last].0.clone();
let va_idx_r = folds[last].1.clone();
let X_tr_r = select_rows(&X_train_z, &tr_idx_r);
let y_tr_r: Vec<f64> = select_y(&y_train_temp, &tr_idx_r);
let X_va_r = select_rows(&X_train_z, &va_idx_r);
let y_va_r: Vec<f64> = select_y(&y_train_temp, &va_idx_r);
let X_tr_r_dm = ndarray_to_dense(&X_tr_r);
let X_va_r_dm = ndarray_to_dense(&X_va_r);

for a in alphas.iter().copied() {
    let model = RidgeRegression::fit(&X_tr_r_dm, &y_tr_r,
        RidgeRegressionParameters::default().with_alpha(a)).unwrap();
    let pred: Vec<f64> = model.predict(&X_va_r_dm).unwrap();
    let r = rmse(y_va_r.as_slice(), pred.as_slice());
    println!("Ridge alpha = {:>8.3} -> CV RMSE = {:.4}", a, r);
    ridge_results.push((a, r));
}

let mut best_ridge: (f64, f64) = ridge_results[0];
for r in ridge_results.iter().copied() {
    if r.1 < best_ridge.1 { best_ridge = r; }
}
println!("\nBest Ridge: alpha = {:.4}, CV RMSE = {:.4}", best_ridge.0, best_ridge.1);

Ridge alpha =    0.010 -> CV RMSE = 2.2078


Ridge alpha =    0.100 -> CV RMSE = 2.2071


Ridge alpha =    1.000 -> CV RMSE = 2.2020


Ridge alpha =   10.000 -> CV RMSE = 2.1904


Ridge alpha =  100.000 -> CV RMSE = 2.2022


Ridge alpha = 1000.000 -> CV RMSE = 2.2607


Best Ridge: alpha = 10.0000, CV RMSE = 2.1904


---
## 5. Grid search — Lasso

In [10]:
// Lasso in smartcore is slow; we use only the last fold.
let lasso_alphas: Vec<f64> = vec![0.001, 0.01, 0.1, 1.0];
let mut lasso_results: Vec<(f64, f64)> = Vec::new();

let last_fold_idx = folds.len() - 1;
let tr_idx_la = folds[last_fold_idx].0.clone();
let va_idx_la = folds[last_fold_idx].1.clone();
let X_tr_la = select_rows(&X_train_z, &tr_idx_la);
let y_tr_la: Vec<f64> = select_y(&y_train_temp, &tr_idx_la);
let X_va_la = select_rows(&X_train_z, &va_idx_la);
let y_va_la: Vec<f64> = select_y(&y_train_temp, &va_idx_la);
let X_tr_la_dm = ndarray_to_dense(&X_tr_la);
let X_va_la_dm = ndarray_to_dense(&X_va_la);

for a in lasso_alphas.iter().copied() {
    let model = Lasso::fit(&X_tr_la_dm, &y_tr_la,
        LassoParameters::default().with_alpha(a)).unwrap();
    let pred: Vec<f64> = model.predict(&X_va_la_dm).unwrap();
    let r = rmse(y_va_la.as_slice(), pred.as_slice());
    println!("Lasso alpha = {:>8.4} -> CV RMSE = {:.4}", a, r);
    lasso_results.push((a, r));
}

let mut best_lasso: (f64, f64) = lasso_results[0];
for r in lasso_results.iter().copied() {
    if r.1 < best_lasso.1 { best_lasso = r; }
}
println!("\nBest Lasso: alpha = {:.4}, CV RMSE = {:.4}", best_lasso.0, best_lasso.1);

Lasso alpha =   0.0010 -> CV RMSE = 2.1921


Lasso alpha =   0.0100 -> CV RMSE = 2.1988


Lasso alpha =   0.1000 -> CV RMSE = 2.1878


Lasso alpha =   1.0000 -> CV RMSE = 2.0955


Best Lasso: alpha = 1.0000, CV RMSE = 2.0955


---
## 6. Grid search — Random Forest Regressor

Expensive: $|n_{trees}| \times |d| \times K_{folds}$ fits. We keep only
4 combinations to fit the notebook time budget.

In [11]:
// 4 combinations to fit the notebook time budget; Nb05 will run the winner.
let rf_combos: Vec<(usize, u16)> = vec![
    (50, 12),
    (50, 20),
    (100, 15),
    (150, 18),
];

// Reuse the last-fold matrices we built above (same train/val split).
let X_tr_rf = select_rows(&X_train, &tr_idx_r);
let X_va_rf = select_rows(&X_val,   &[]);  // placeholder, we use last fold train/val below
let X_tr_rf = select_rows(&X_train, &tr_idx_r);
let X_va_rf = select_rows(&X_train, &va_idx_r);
let X_tr_rf_dm = ndarray_to_dense(&X_tr_rf);
let X_va_rf_dm = ndarray_to_dense(&X_va_rf);

let mut rf_results: Vec<((usize, u16), f64)> = Vec::new();
for &(n_t, d) in &rf_combos {
    let model = RandomForestRegressor::fit(&X_tr_rf_dm, &y_tr_r,
        RandomForestRegressorParameters::default().with_n_trees(n_t).with_max_depth(d)).unwrap();
    let pred: Vec<f64> = model.predict(&X_va_rf_dm).unwrap();
    let r = rmse(y_va_r.as_slice(), pred.as_slice());
    println!("RF (n_trees={:>3}, depth={:>2}) -> CV RMSE = {:.4}", n_t, d, r);
    rf_results.push(((n_t, d), r));
}

let mut best_rf: ((usize, u16), f64) = rf_results[0].clone();
for r in rf_results.iter() {
    if r.1 < best_rf.1 { best_rf = r.clone(); }
}
println!("\nBest RF: n_trees={}, depth={}, CV RMSE = {:.4}",
         best_rf.0.0, best_rf.0.1, best_rf.1);

RF (n_trees= 50, depth=12) -> CV RMSE = 2.3416


RF (n_trees= 50, depth=20) -> CV RMSE = 2.3072


RF (n_trees=100, depth=15) -> CV RMSE = 2.3045


RF (n_trees=150, depth=18) -> CV RMSE = 2.2815


Best RF: n_trees=150, depth=18, CV RMSE = 2.2815


---
## 7. Random search — Gradient Boosting

Search space:
- $K \in [40, 200]$ (number of trees)
- $d \in [3, 8]$ (depth)
- $\eta \in [0.03, 0.30]$ (learning rate)

12 samples from `StdRng(seed=42)`. For each combination we run **early
stopping**: stop if val RMSE does not improve for 10 consecutive
iterations.

In [12]:
let mut rng = StdRng::seed_from_u64(42);
let n_iter = 12;
let mut gb_results: Vec<((usize, u16, f64), f64)> = Vec::new();

for trial in 0..n_iter {
    let k_max: usize = rng.gen_range(40..=200);
    let depth: u16 = rng.gen_range(3..=8) as u16;
    let eta:   f64 = rng.gen_range(3..=30) as f64 / 100.0;

    // We evaluate GB on a single fold (the last one) because random search
    // is already sampling -- no need to also average across folds.
    let (tr_idx, va_idx) = (tr_idx_r.clone(), va_idx_r.clone());
    let X_tr_dm = X_tr_rf_dm.clone();
    let X_va_dm = X_va_rf_dm.clone();
    let y_tr = y_tr_r.clone();
    let y_va = y_va_r.clone();

    let n_train_f = y_tr.len();
    let n_val_f   = y_va.len();
    let init = y_tr.iter().sum::<f64>() / n_train_f as f64;
    let mut tr_pred = vec![init; n_train_f];
    let mut va_pred = vec![init; n_val_f];

    let mut best_rmse = f64::MAX;
    let mut best_iter = 0usize;
    let mut patience = 0usize;
    let mut final_rmse = f64::MAX;

    for k in 0..k_max {
        let res: Vec<f64> = y_tr.iter().zip(tr_pred.iter()).map(|(a,b)| a-b).collect();
        let tree = DecisionTreeRegressor::fit(&X_tr_dm, &res,
            DecisionTreeRegressorParameters::default().with_max_depth(depth)).unwrap();
        let upd_tr: Vec<f64> = tree.predict(&X_tr_dm).unwrap();
        let upd_va: Vec<f64> = tree.predict(&X_va_dm).unwrap();
        for i in 0..n_train_f { tr_pred[i] += eta * upd_tr[i]; }
        for i in 0..n_val_f   { va_pred[i] += eta * upd_va[i]; }
        let r = rmse(y_va.as_slice(), va_pred.as_slice());
        if r < best_rmse {
            best_rmse = r; best_iter = k+1; patience = 0;
        } else {
            patience += 1;
            if patience >= 10 { final_rmse = best_rmse; break; }
        }
        final_rmse = best_rmse;
    }
    println!("trial {:>2}: K={:>3}, d={}, eta={:.2} -> CV RMSE = {:.4} (best @ iter {})",
             trial+1, k_max, depth, eta, final_rmse, best_iter);
    gb_results.push(((best_iter, depth, eta), final_rmse));
}

let mut best_gb: ((usize, u16, f64), f64) = gb_results[0].clone();
for r in gb_results.iter() {
    if r.1 < best_gb.1 { best_gb = r.clone(); }
}
println!("\nBest GB: K={}, d={}, eta={:.2} -> CV RMSE = {:.4}",
         best_gb.0.0, best_gb.0.1, best_gb.0.2, best_gb.1);

trial  1: K=127, d=8, eta=0.20 -> CV RMSE = 2.2193 (best @ iter 17)


trial  2: K=105, d=3, eta=0.20 -> CV RMSE = 2.2452 (best @ iter 61)


trial  3: K=141, d=3, eta=0.26 -> CV RMSE = 2.2991 (best @ iter 60)


trial  4: K=136, d=6, eta=0.27 -> CV RMSE = 2.2082 (best @ iter 11)


trial  5: K=154, d=6, eta=0.09 -> CV RMSE = 2.1867 (best @ iter 41)


...

trial  9: K= 88, d=5, eta=0.25 -> CV RMSE = 2.2140 (best @ iter 35)


trial 10: K=145, d=8, eta=0.14 -> CV RMSE = 2.4362 (best @ iter 23)


trial 11: K=183, d=8, eta=0.28 -> CV RMSE = 2.3125 (best @ iter 9)


trial 12: K=144, d=4, eta=0.19 -> CV RMSE = 2.2080 (best @ iter 41)


Best GB: K=41, d=6, eta=0.09 -> CV RMSE = 2.1867


---
## 8. Grid search — RandomForestClassifier

Best classifier from Nb03. We tune $(n_{trees}, max\_depth)$.

In [13]:
let rfc_combos: Vec<(usize, u16)> = vec![
    (50, 12), (100, 15), (150, 18), (200, 20),
];

let mut rfc_results: Vec<((usize, u16), f64)> = Vec::new();
for &(n_t, d) in &rfc_combos {
    let mut fold_mccs = Vec::new();
    for (tr_idx, va_idx) in &folds {
        let X_tr = select_rows(&X_train, tr_idx);
        let y_tr = select_yu(&y_train_rain, tr_idx);
        let X_va = select_rows(&X_train, va_idx);
        let y_va = select_yu(&y_train_rain, va_idx);
        let X_tr_dm = ndarray_to_dense(&X_tr);
        let X_va_dm = ndarray_to_dense(&X_va);
        let model = RandomForestClassifier::fit(&X_tr_dm, &y_tr,
            RandomForestClassifierParameters::default().with_n_trees(n_t as u16).with_max_depth(d)).unwrap();
        let pred: Vec<u32> = model.predict(&X_va_dm).unwrap();
        fold_mccs.push(mcc(&y_va, &pred));
    }
    let mean = fold_mccs.iter().sum::<f64>() / fold_mccs.len() as f64;
    println!("RFC (n_trees={:>3}, depth={:>2}) -> CV MCC = {:.4}", n_t, d, mean);
    rfc_results.push(((n_t, d), mean));
}

let mut best_rfc: ((usize, u16), f64) = rfc_results[0].clone();
for r in rfc_results.iter() {
    if r.1 > best_rfc.1 { best_rfc = r.clone(); }
}
println!("\nBest RFC: n_trees={}, depth={}, CV MCC = {:.4}",
         best_rfc.0.0, best_rfc.0.1, best_rfc.1);

RFC (n_trees= 50, depth=12) -> CV MCC = 0.4541


RFC (n_trees=100, depth=15) -> CV MCC = 0.4599


RFC (n_trees=150, depth=18) -> CV MCC = 0.4617


RFC (n_trees=200, depth=20) -> CV MCC = 0.4621


Best RFC: n_trees=200, depth=20, CV MCC = 0.4621


---
## 9. Logistic Regression

Smartcore 0.3 exposes limited regularization control over
`LogisticRegression` (there is no direct `with_alpha`). We keep the
default configuration and measure its CV MCC for a fair comparison.

In [14]:
let mut lr_mccs = Vec::new();
for (tr_idx, va_idx) in &folds {
    let X_tr = select_rows(&X_train_z, tr_idx);
    let y_tr = select_yu(&y_train_rain, tr_idx);
    let X_va = select_rows(&X_train_z, va_idx);
    let y_va = select_yu(&y_train_rain, va_idx);
    let X_tr_dm = ndarray_to_dense(&X_tr);
    let X_va_dm = ndarray_to_dense(&X_va);
    let model = LogisticRegression::fit(&X_tr_dm, &y_tr, LogisticRegressionParameters::default()).unwrap();
    let pred: Vec<u32> = model.predict(&X_va_dm).unwrap();
    lr_mccs.push(mcc(&y_va, &pred));
}
let lr_mean = lr_mccs.iter().sum::<f64>() / lr_mccs.len() as f64;
println!("LogisticRegression CV MCC = {:.4}", lr_mean);

LogisticRegression CV MCC = 0.4332


---
## 10. Learning curves (best Ridge)

We train the best Ridge on fractions $\{0.3, 0.5, 0.7, 1.0\}$ of the
train set and measure RMSE on (i) the train itself and (ii) the stable
val holdout. Diagnoses overfitting vs underfitting.

In [15]:
// Fractions starting at 0.3 - smaller fractions cause ill-conditioning.
let train_sizes = [0.3_f64, 0.5, 0.7, 1.0];
let n_train = X_train_z.nrows();
let mut lc: Vec<(f64, f64, f64)> = Vec::new();

for f in train_sizes.iter().copied() {
    let n_use = (n_train as f64 * f) as usize;
    let X_sub = X_train_z.slice(ndarray::s![..n_use, ..]).to_owned();
    let y_sub: Vec<f64> = y_train_temp.slice(ndarray::s![..n_use]).to_vec();
    let X_sub_dm = ndarray_to_dense(&X_sub);
    let X_val_dm = ndarray_to_dense(&X_val_z);
    let y_val_v: Vec<f64> = y_val_temp.to_vec();
    let m = RidgeRegression::fit(&X_sub_dm, &y_sub,
        RidgeRegressionParameters::default().with_alpha(best_ridge.0)).unwrap();
    let pt: Vec<f64> = m.predict(&X_sub_dm).unwrap();
    let pv: Vec<f64> = m.predict(&X_val_dm).unwrap();
    let r_tr = rmse(y_sub.as_slice(), pt.as_slice());
    let r_va = rmse(y_val_v.as_slice(), pv.as_slice());
    println!("frac={:.1} ({:>5} samples) -> train={:.4}  val={:.4}", f, n_use, r_tr, r_va);
    lc.push((f, r_tr, r_va));
}

let final_gap = lc.last().unwrap().2 - lc.last().unwrap().1;
println!("\nGap (val - train) at 100% data: {:.4}", final_gap);
if final_gap > 0.5 { println!("-> mild overfitting; more regularization or more data might help."); }
else if lc.last().unwrap().2 > 4.0 { println!("-> underfitting; consider extra features or non-linear models."); }
else { println!("-> model is well-balanced."); }

frac=0.3 ( 5846 samples) -> train=2.5986  val=12467.9990


frac=0.5 ( 9744 samples) -> train=2.6311  val=3.8242


frac=0.7 (13641 samples) -> train=2.5921  val=2.5414


frac=1.0 (19488 samples) -> train=2.4531  val=2.5032


Gap (val - train) at 100% data: 0.0501


-> model is well-balanced.


()

---
## 11. Persist best hyperparameters

In [16]:
use serde_json::json;

let bundle = json!({
    "regression": {
        "ridge": { "alpha": best_ridge.0, "cv_rmse": best_ridge.1 },
        "lasso": { "alpha": best_lasso.0, "cv_rmse": best_lasso.1 },
        "random_forest": { "n_trees": best_rf.0.0, "max_depth": best_rf.0.1, "cv_rmse": best_rf.1 },
        "gradient_boosting": {
            "n_trees": best_gb.0.0, "max_depth": best_gb.0.1, "learning_rate": best_gb.0.2,
            "cv_rmse": best_gb.1
        }
    },
    "classification": {
        "random_forest": { "n_trees": best_rfc.0.0, "max_depth": best_rfc.0.1, "cv_mcc": best_rfc.1 },
        "logistic": { "cv_mcc": lr_mean }
    },
    "selected_target": "temp_next_24h",
    "cv_strategy": "3-fold forward chaining (last fold used for regression)"
});

std::fs::create_dir_all("../models").unwrap();
std::fs::write("../models/best_hyperparameters.json",
    serde_json::to_string_pretty(&bundle).unwrap()).unwrap();
println!("Saved ../models/best_hyperparameters.json");

Saved ../models/best_hyperparameters.json


---
## 12. Summary

| model | tuned hyperparameter | CV |
|---|---|---|
| Ridge | $\alpha$ | RMSE |
| Lasso | $\alpha$ | RMSE |
| RandomForest regressor | $(n_t, d)$ | RMSE |
| GradientBoosting | $(K, d, \eta)$ | RMSE |
| RandomForest classifier | $(n_t, d)$ | MCC |

-> Next: **Notebook 05** — final rigorous evaluation on the test set with
per-city / season / temperature-bin breakdown, error distribution,
classifier calibration, and skill scores relative to strong baselines.

In [17]:
println!("\n{}", "=".repeat(60));
println!("Notebook 04 complete.");
println!("{}", "=".repeat(60));
println!("Next: Notebook 05 - Evaluation & Validation");

Notebook 04 complete.


Next: Notebook 05 - Evaluation & Validation
